# A7a: chạy giải pháp nộp bằng trọng số THẬT trên Kaggle, không có Docker (TASKS mục A7a; `scripts/verify_image.py --no-docker`). Độc lập với A5. Khoảng **1 đến 1,5 giờ GPU**.
Chạy cùng đường mã `run_container.py` như bản nộp: **ASR sống** (`phowl_ft/merged`) + Qwen3-4B + adapter `q4b_lora` + luật, offline, trên 634 mẫu `public_test`, **3 lần**. Kiểm tra: ba lần trùng từng byte, `verify_submission` báo 634/634 hash khớp, mẫu chậm nhất (`max_s`) dưới 30 giây, và số dòng lệch so với bài nộp tham chiếu `sub_public_v1.jsonl` (kỳ vọng 0 lệch).

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**.
**Input cần gắn:**
1. dataset `lifemate-code` bản mới từ commit `78c70a3` (có `verify_image.py`)
2. dataset `train+public_test` (raw; cần `public_test/` có audio và `zaic_tools`)
3. dataset `thuan-qwen3-4b`
4. **output của notebook `Hello_done`** (adapter `q4b_lora` và ASR `phowl_ft/merged`)
5. *(nên có)* dataset **Private** chứa file `sub_public_v1.jsonl` (bài nộp public lần 1, 634 dòng) để so từng dòng. Thiếu thì A-1 báo và vẫn chạy, chỉ không có phép so này.

**Thứ tự cell:** P-1 đến P-3, A-1 (kiểm tra, sha256 so với `models.lock.json`, vá `-I` trong bản sao `verify_image.py`), A-2 (thử nhanh 20 mẫu × 2 lần), A-3 (đầy đủ), A-4 (đóng zip), A-5 (dọn, chạy cuối rồi Save Version). **Gửi Phương:** `a7a_results.zip` (`report.json` không chứa nội dung dữ liệu). Nếu A-2 lỗi thì dừng và báo **nguyên văn** lỗi.

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = Qwen3-4B (hidden 2560, 36 tầng), tìm theo cấu hình vì còn có Qwen3-8B/Qwen3.5-9B trong input (không lấy "qwen3 đầu tiên")
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
QW=$(python -X utf8 - <<'EOF'
import json
from pathlib import Path
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    t = c.get("text_config", c)
    if c.get("model_type") == "qwen3" and t.get("hidden_size") == 2560 and t.get("num_hidden_layers") == 36:
        print(p.parent)
        break
EOF
)
echo "Qwen3-4B dir: ${QW:-KHÔNG THẤY}"
[ -n "$QW" ] || { echo "ERROR: không thấy Qwen3-4B (hidden 2560, 36 tầng). Gắn dataset thuan-qwen3-4b."; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ output notebook trước? (checkpoints và outputs)"; ls /kaggle/working/checkpoints | head -20; ls /kaggle/working/outputs | head -12

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn đoạn mã đóng gói (chỉ file kết quả nhỏ, không có nội dung dữ liệu)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# A-1: kiểm tra đủ đồ cho A7a (chạy giải pháp nộp bằng trọng số THẬT), đối chiếu sha256 adapter và ASR với models.lock.json, tìm bài nộp tham chiếu.
# Thiếu đồ thì DỪNG. Rồi dọn bản sao không dùng và vá `-I` trong BẢN SAO verify_image.py trên Kaggle (xem ghi chú dưới).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/verify_image.py
need scripts/run_container.py
need scripts/verify_submission.py
need src/lifemate/container.py
need models.lock.json
need checkpoints/q4b_lora/adapter_model.safetensors
need checkpoints/q4b_lora/adapter_config.json
need checkpoints/phowl_ft/merged/config.json
need "data/raw/train+public_test/public_test/public_test.jsonl"
[ -d "data/raw/train+public_test/public_test/audio" ] && echo "ok     public_test/audio" || { echo "THIẾU  public_test/audio"; MISSING=1; }
[ -d "data/raw/train+public_test/public_test/zaic_tools" ] && echo "ok     public_test/zaic_tools" || { echo "THIẾU  public_test/zaic_tools"; MISSING=1; }
if [ "$MISSING" = 1 ]; then echo "-> Kiểm tra input: lifemate-code (bản từ 78c70a3), train+public_test, thuan-qwen3-4b, output notebook Hello_done."; exit 1; fi
echo "-- trọng số ASR đã gộp:"; ls -la checkpoints/phowl_ft/merged | head -12
echo "-- đối chiếu sha256 với models.lock.json (không khớp thì báo Phương; vẫn chạy tiếp để lấy số thời gian)"
python -X utf8 - <<'EOF'
import hashlib, json
from pathlib import Path
lock = json.load(open("models.lock.json", encoding="utf-8"))
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(1 << 24):
            h.update(b)
    return h.hexdigest()
for key, path in (("adapter", "checkpoints/q4b_lora"), ("asr", "checkpoints/phowl_ft/merged")):
    for name, want in lock[key]["weights"].items():
        p = Path(path) / name
        if not p.is_file():
            print(f"{key}: THIẾU {p}"); continue
        got = sha(p)
        print(f"{key}: {name} {'TRÙNG models.lock.json' if got == want else 'KHÁC models.lock.json'} ({got[:12]}… so với {want[:12]}…)")
EOF
echo "-- bài nộp tham chiếu (sub_public_v1.jsonl, để so từng dòng):"
REF=$(find /kaggle/input -maxdepth 6 -name sub_public_v1.jsonl -print -quit 2>/dev/null)
if [ -n "$REF" ]; then echo "export REF=$REF" > /kaggle/working/a7_env.sh; wc -l "$REF"; else echo "export REF=" > /kaggle/working/a7_env.sh; echo "KHÔNG thấy sub_public_v1.jsonl trong input: vẫn chạy được nhưng KHÔNG có phép so với bài nộp (gắn dataset chứa file đó rồi chạy lại cell này)"; fi
echo "-- dọn bản sao không dùng (chỉ giữ q4b_lora và phowl_ft/merged)"
for d in checkpoints/*; do case "$(basename "$d")" in q4b_lora|phowl_ft) ;; *) rm -rf "$d" ;; esac; done
rm -rf checkpoints/q4b_lora/checkpoint-*
for d in checkpoints/phowl_ft/*; do [ "$(basename "$d")" = "merged" ] || rm -rf "$d"; done
rm -rf /kaggle/working/outputs/*
mkdir -p outputs/logs
du -sh checkpoints | cut -f1 | xargs echo "checkpoints sau khi dọn:"
echo "-- vá BẢN SAO verify_image.py trên Kaggle: bỏ cờ -I khi nó gọi Python con (như các .sh đã được vá bằng sed; mã trong repo KHÔNG đổi)"
grep -c '"-I", "-X", "utf8"' scripts/verify_image.py | xargs echo "số chỗ có -I trước khi vá:"
sed -i 's/"-I", "-X", "utf8"/"-X", "utf8"/g' scripts/verify_image.py
grep -c '"-I", "-X", "utf8"' scripts/verify_image.py | xargs echo "số chỗ có -I sau khi vá:"
df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A-2 (khoảng 3 đến 6 phút): chạy THỬ NHANH 20 mẫu, 2 lần (đúng gợi ý "thử nhanh trước" trong docs/DOCKER.md) để biết đường chạy thông. Lỗi thì dừng, báo NGUYÊN VĂN.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a7_env.sh
REFARG=""; [ -n "${REF:-}" ] && REFARG="--ref $REF"
python -X utf8 scripts/verify_image.py --no-docker --data-root data/raw/train+public_test --split public_test --runs 2 --limit 20 \
  --out-dir /kaggle/working/check_a7a_quick $REFARG -- --llm "$QWEN" --adapter checkpoints/q4b_lora --asr checkpoints/phowl_ft/merged 2>&1 | tail -n 25
CODE=${PIPESTATUS[0]}
echo; echo "exit: $CODE (0 = ALL CHECKS PASSED)"
ls /kaggle/working/check_a7a_quick 2>/dev/null

In [ ]:
%%bash
# A-3 (khoảng 50 đến 90 phút): A7a ĐẦY ĐỦ: 3 lần chạy 634 mẫu public_test bằng đúng đường mã của container (ASR sống + LLM + luật, offline), rồi verify_submission,
# so byte giữa 3 lần, đo thời gian, so từng dòng với sub_public_v1.jsonl. Đúng lệnh của docs/TASKS.md mục A7a.
# Cell này KHÔNG dừng khi lỗi: để cell đóng gói vẫn lấy được report.json.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a7_env.sh
REFARG=""; [ -n "${REF:-}" ] && REFARG="--ref $REF"
START=$(date +%s)
python -X utf8 scripts/verify_image.py --no-docker --data-root data/raw/train+public_test --split public_test --runs 3 \
  --out-dir /kaggle/working/check_a7a $REFARG -- --llm "$QWEN" --adapter checkpoints/q4b_lora --asr checkpoints/phowl_ft/merged 2>&1 | tee outputs/logs/a7a_verify_image.txt | tail -n 30
CODE=${PIPESTATUS[0]}
echo "verify_image exit code: $CODE (0 = ALL CHECKS PASSED)"
echo "tổng thời gian: $(( $(date +%s) - START )) s" | tee -a outputs/logs/a7a_verify_image.txt
ls -la /kaggle/working/check_a7a

In [ ]:
%%bash
# A-4: gói kết quả để gửi Phương: a7a_results.zip. report.json KHÔNG chứa nội dung dữ liệu (chỉ GPU, thời gian, sha256, số dòng lệch). Không đưa submission.jsonl vào zip.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
echo "== report.json (đầy đủ) =="
python -X utf8 -c "
import json
r = json.load(open('/kaggle/working/check_a7a/report.json', encoding='utf-8'))
print(json.dumps({k: v for k, v in r.items() if k != 'runs'}, ensure_ascii=False, indent=1))
for e in r['runs']: print(e)
" 2>&1 | cut -c1-300
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a7a_results.zip \
  /kaggle/working/check_a7a/report.json /kaggle/working/check_a7a_quick/report.json outputs/logs/a7a_*.txt /kaggle/working/torch_cuda.txt
ls -la /kaggle/working/*.zip

In [ ]:
%%bash
# A-5 (CUỐI CÙNG): dọn (xóa lifemate/ và bản sao trọng số), giữ check_a7a/, zip và log. Rồi Save Version.
cd /kaggle/working
rm -rf lifemate helpers setup_patched.sh config_paths.txt checkpoints state
du -sh /kaggle/working/* 2>/dev/null
ls /kaggle/working
echo "xong. Bây giờ Save Version (private). Lấy a7a_results.zip ở tab Output."